# Symbolic Differentiation 

In this notebook our goal is to implement *symbolic differentiation*.  Concretely, we want to implement a function `diff` that takes one argument:
  - The argument `expr` represents an *arithmetic expression*.
    Here an arithmetic expression is any string that is build from variable and numbers by application
    of any of the operator symbols "`+`", "`-`", "`*`", "`/`", and "`**`".
    The operator "`**`" represents exponentiation, i.e. an expression of the form 
    $a \texttt{**} b$ is interpreted as $a^b$.          
  - If $e$ is an expression, then both $\exp(e)$ and $\ln(e)$ are expressions too.

The function call `diff(expr)` will then compute the derivative of `expr` with respect to the variable `x`.  For example, the function call 
`diff("x * exp(x)")` will compute the output
`1 * exp(x) + x * exp(x)` because we have:
$$ \frac{\mathrm{d}\;}{\mathrm{d}x} \bigl( x \cdot \mathrm{e}^x \bigr) = 1 \cdot \mathrm{e}^x + x \cdot \mathrm{e}^x. $$

This file shows the implementation of a program that can perform *symbolic differentiation* using `Larkita`.
The parser generated by `Larkita` returns
nested tuples, so our symbolic differentiation function can leverage pattern matching
right away.

## Specification of the Grammar and Parser

We use `Larkita` to define our EBNF grammar, which is stored in the file `diff.lark`.  The rules inherently
enforce mathematical precedence.  We include a
rule `unary` to handle unary minus correctly relative to exponentiation and multiplication.

The string literals of an alternative form the *head* of the tuple that is returned for this alternative, while
the other symbols become its children.  For example, the alternative `expr "+" product` yields the tuple
`("+", ast(expr), ast(product))`, i.e. `Node [Leaf (Str "+"); ast(expr); ast(product)]`, and `base "**" unary`
yields `("**", ast(base), ast(unary))`.  An *alias* `-> name` replaces the head, so the alternative
`"-" unary -> neg` yields `("neg", ast(unary))` and the alternative `"exp" "(" expr ")" -> exp` yields
`("exp", ast(expr))`.  An alternative that consists of a single symbol, like
`product: unary`, returns the abstract syntax tree of this symbol.  The directive `%ignore "(", ")"` removes the
parentheses from the head of the alternative `"(" expr ")"`, which then returns the abstract syntax tree of
`expr`.

In [ ]:
#use "../lib/nb.ml";;
load_lib "larkita.ml"

In [ ]:
cat "diff.lark"

We generate a parser for this grammar.  `Larkita` writes a description of the states of the LR(1) automaton into
the file `diff.out`.

In [ ]:
let parser = Larkita.open_ ~out_file:"diff.out" "diff.lark"

## AST Construction

The parser already returns nested tuples whose heads are the operators:

In [ ]:
Larkita.parse parser "ln(x ** x) + exp(x * -x)"

The function `convert_leaves` replaces the *leaves* of these tuples:  a number like `("NUMBER", "2")` becomes the
integer leaf `2`, i.e. `Leaf (Int 2)`, and the variable `x`, which is a string literal and therefore the tuple
`("x",)`, becomes the string `"x"`.  All other tuples are converted recursively.  The function `diff` then works on
the nested tuples, e.g. `("+", ("ln", "x"), 1)`.

The helper functions `op1` and `op2` build tuples consisting of an operator and one or two arguments.

In [ ]:
let op1 (op : string) (a : nested) : nested = Node [ Leaf (Str op); a ]

let op2 (op : string) (a : nested) (b : nested) : nested = Node [ Leaf (Str op); a; b ]

In [ ]:
let rec convert_leaves (ast : nested) : nested =
  match ast with
  | Node [ Leaf (Str "NUMBER"); Leaf (Str digits) ] -> Leaf (Int (int_of_string digits))
  | Node [ Leaf (Str "x") ] -> Leaf (Str "x")
  | Node (op :: args) -> Node (op :: List.map convert_leaves args)
  | _ -> ast

In [ ]:
let parse (s : string) : nested = convert_leaves (Larkita.parse parser s)

In [ ]:
parse "ln(x ** x) + exp(x * -x)"

## Symbolic Differentiation

Now we are ready to implement the function `diff` that takes an expression possibly containing the variable `x`.
It computes the derivative of the given expression with respect to `x`.
<ol>
<li>  The first case implements the rule:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) + g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) + \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The second case implements the rule:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(f(x) - g(x)\bigr) = \frac{\mathrm{d}\;}{\mathrm{d}x} f(x) - \frac{\mathrm{d}\;}{\mathrm{d}x} g(x)$$
      </li>
<li>  The third case deals with the product rule.
      </li>
<li>  The fourth case deals with the quotient rule.
      </li>
<li>  The fifth case deals with exponentiation trick (rewriting using ln and exp).
      </li>
<li>  The sixth case deals with unary negation:
      $$\frac{\mathrm{d}\;}{\mathrm{d}x}\bigl(-f(x)\bigr) = -\frac{\mathrm{d}\;}{\mathrm{d}x} f(x)$$
      </li>
<li>  The seventh case deals with natural logarithm.
      </li>
<li>  The eighth case deals with the exponential function.
      </li>
<li>  The derivative of `x` is 1.  Otherwise, the expression is a constant and hence we return 0.
      </li>
</ol>

In [ ]:
let rec diff (e : nested) : nested =
  (* differentiate the expression e with respect to the variable x *)
  match e with
  | Node [ Leaf (Str "+"); f; g ] ->
      let fs, gs = diff f, diff g in
      op2 "+" fs gs
  | Node [ Leaf (Str "-"); f; g ] ->
      let fs, gs = diff f, diff g in
      op2 "-" fs gs
  | Node [ Leaf (Str "*"); f; g ] ->
      let fs, gs = diff f, diff g in
      op2 "+" (op2 "*" fs g) (op2 "*" f gs)
  | Node [ Leaf (Str "/"); f; g ] ->
      let fs, gs = diff f, diff g in
      op2 "/" (op2 "-" (op2 "*" fs g) (op2 "*" f gs)) (op2 "*" g g)
  | Node [ Leaf (Str "**"); f; g ] -> diff (op1 "exp" (op2 "*" g (op1 "ln" f)))
  | Node [ Leaf (Str "neg"); f ] -> op1 "neg" (diff f)
  | Node [ Leaf (Str "ln"); f ] ->
      let fs = diff f in
      op2 "/" fs f
  | Node [ Leaf (Str "exp"); f ] ->
      let fs = diff f in
      op2 "*" fs e
  | Leaf (Str "x") -> Leaf (Int 1)
  | _ -> Leaf (Int 0)

### Arithmetic with Numbers

The simplification rules below compute with numbers.  A number is either an integer `Int n` or a floating point
number `Float x`.  The following functions implement the arithmetic:
- The result of `+`, `-`, and `*` is an integer if both arguments are integers and a floating point number otherwise.
- The result of `/` is always a floating point number.
- The result of `**` is an integer if both arguments are integers and the exponent is not negative.
- Integers and floating point numbers are compared by their values, e.g. `1` and `1.0` are equal.  The function
  `equal` compares nested tuples in the same way.

In [ ]:
let is_number (e : nested) : bool =
  match e with
  | Leaf (Int _ | Float _) -> true
  | _ -> false

let to_float (n : value) : float =
  match n with
  | Int k -> float_of_int k
  | Float x -> x
  | _ -> invalid_arg "to_float: not a number"

let num_eq (a : value) (b : value) : bool = to_float a = to_float b

let arith (int_op : int -> int -> int) (float_op : float -> float -> float) (a : value) (b : value) : value =
  match a, b with
  | Int x, Int y -> Int (int_op x y)
  | _ -> Float (float_op (to_float a) (to_float b))

let num_add = arith ( + ) ( +. )
let num_sub = arith ( - ) ( -. )
let num_mul = arith ( * ) ( *. )
let num_div (a : value) (b : value) : value = Float (to_float a /. to_float b)

let num_pow (a : value) (b : value) : value =
  match a, b with
  | Int x, Int y when y >= 0 -> Int (int_of_float (Float.pow (float_of_int x) (float_of_int y)))
  | _ -> Float (Float.pow (to_float a) (to_float b))

let num_neg (a : value) : value =
  match a with
  | Int x -> Int (-x)
  | _ -> Float (-.(to_float a))

(* is_value k e checks whether e is the number k *)
let is_value (k : int) (e : nested) : bool =
  match e with
  | Leaf ((Int _ | Float _) as n) -> num_eq n (Int k)
  | _ -> false

let rec equal (e1 : nested) (e2 : nested) : bool =
  match e1, e2 with
  | Leaf ((Int _ | Float _) as a), Leaf ((Int _ | Float _) as b) -> num_eq a b
  | Node l1, Node l2 -> List.length l1 = List.length l2 && List.for_all2 equal l1 l2
  | _ -> e1 = e2

### Summary of Simplification Rules

The `simplify` function applies the following algebraic identities to transform expression trees into their canonical, reduced forms. Simplification is performed bottom-up, with recursive re-evaluation to ensure chained simplifications (e.g., `--a -> a`) are fully applied.

#### 1. Constant Folding
* All arithmetic operations (`+`, `-`, `*`, `/`, `**`) are evaluated whenever both operands are numeric constants.

#### 2. Unary Negation Folding
* $-(-a) \rightarrow a$
* $-0 \rightarrow 0$
* $-a$ (where $a$ is a constant) $\rightarrow$ numeric negation

#### 3. Addition & Subtraction Identities
* $a + 0 = a$
* $0 - a = -a$
* $a - 0 = a$
* $a - a = 0$
* $a + (-b) \rightarrow a - b$
* $-a + b \rightarrow b - a$
* $a - (-b) \rightarrow a + b$

#### 4. Multiplication & Reciprocal Rules
* $a \cdot 0 = 0$
* $a \cdot 1 = a$
* $-1 \cdot a \rightarrow -a$
* $a \cdot \frac{1}{a} = 1$
* $-a \cdot \frac{1}{a} = -1$
* $a \cdot (-\frac{1}{a}) = -1$

#### 5. Power Derivative Reduction
*Used to simplify chain rule outputs:*
* $a \cdot \frac{1}{f} \cdot f^b \rightarrow a \cdot f^{b-1}$
* $\frac{1}{f} \cdot f^b \rightarrow f^{b-1}$
* *(Special Case $b=2$)*: $a \cdot \frac{1}{f} \cdot f^2 \rightarrow a \cdot f$

#### 6. Exponentiation, Logarithmic & Exponential Identities
* $a^0 = 1$, $a^1 = a$, $0^a = 0$, $1^a = 1$
* $\ln(1) = 0$
* $\ln(\exp(a)) = a$
* $\exp(0) = 1$
* $\exp(\ln(a)) = a$
* $\exp(a \cdot \ln(f)) \rightarrow f^a$

The cases are tried from top to bottom and the first case that matches is used.  A rule that contains a number,
e.g. $0 + a = a$, becomes a pattern with a condition, e.g. `Node [Leaf (Str "+"); z; a] when is_value 0 z`, since
the number `0` might be the integer `0` or the floating point number `0.0`.  For the same reason, two subexpressions
are compared with the function `equal`.

The function `simplify` simplifies the arguments of an operator first.  Then the function `rules` applies the
simplification rules to the result.  Both functions call each other.

In [ ]:
let one  : nested = Leaf (Int 1)
let zero : nested = Leaf (Int 0)

let rec simplify (e : nested) : nested =
  (* Recursively simplifies an abstract syntax tree representing an arithmetic expression. *)
  match e with
  (* 1. Recursively simplify sub-expressions first (bottom-up) *)
  | Node [ op; left; right ] -> rules (Node [ op; simplify left; simplify right ])
  | Node [ op; arg ] -> rules (Node [ op; simplify arg ])
  | _ -> e

and rules (e_sim : nested) : nested =
  (* 2. Apply algebraic simplification rules *)
  match e_sim with
  (* Unary Negation Folding *)
  | Node [ Leaf (Str "neg"); Leaf ((Int _ | Float _) as a) ] -> Leaf (num_neg a)
  | Node [ Leaf (Str "neg"); Node [ Leaf (Str "neg"); a ] ] -> a
  | Node [ Leaf (Str "neg"); z ] when is_value 0 z -> zero

  (* Constant Folding *)
  | Node [ Leaf (Str op); Leaf ((Int _ | Float _) as a); Leaf ((Int _ | Float _) as b) ] -> (
      match op with
      | "+" -> Leaf (num_add a b)
      | "-" -> Leaf (num_sub a b)
      | "*" -> Leaf (num_mul a b)
      | "/" when not (num_eq b (Int 0)) -> Leaf (num_div a b)
      | "**" -> Leaf (num_pow a b)
      | _ -> e_sim)

  (* Addition & Subtraction *)
  | Node [ Leaf (Str "+"); z; a ] when is_value 0 z -> a
  | Node [ Leaf (Str "+"); a; z ] when is_value 0 z -> a
  | Node [ Leaf (Str "+"); a; Node [ Leaf (Str "neg"); b ] ] -> simplify (op2 "-" a b)
  | Node [ Leaf (Str "+"); Node [ Leaf (Str "neg"); a ]; b ] -> simplify (op2 "-" b a)

  (* Force re-evaluation on new structures *)
  | Node [ Leaf (Str "-"); z; a ] when is_value 0 z -> simplify (op1 "neg" a)
  | Node [ Leaf (Str "-"); a; Node [ Leaf (Str "neg"); b ] ] -> simplify (op2 "+" a b)

  | Node [ Leaf (Str "-"); a; z ] when is_value 0 z -> a
  | Node [ Leaf (Str "-"); a; b ] when equal a b -> zero

  (* Multiplication *)
  | Node [ Leaf (Str "*"); z; _ ] when is_value 0 z -> zero
  | Node [ Leaf (Str "*"); _; z ] when is_value 0 z -> zero
  | Node [ Leaf (Str "*"); o; a ] when is_value 1 o -> a
  | Node [ Leaf (Str "*"); a; o ] when is_value 1 o -> a

  (* Force re-evaluation on new structures *)
  | Node [ Leaf (Str "*"); m; a ] when is_value (-1) m -> simplify (op1 "neg" a)
  | Node [ Leaf (Str "*"); a; m ] when is_value (-1) m -> simplify (op1 "neg" a)

  | Node [ Leaf (Str "*"); a; Node [ Leaf (Str "/"); o; b ] ] when is_value 1 o && equal a b -> one
  | Node [ Leaf (Str "*"); Node [ Leaf (Str "/"); o; b ]; a ] when is_value 1 o && equal a b -> one
  | Node [ Leaf (Str "*"); Node [ Leaf (Str "neg"); a ]; Node [ Leaf (Str "/"); o; b ] ]
    when is_value 1 o && equal a b -> Leaf (Int (-1))
  | Node [ Leaf (Str "*"); Node [ Leaf (Str "/"); o; b ]; Node [ Leaf (Str "neg"); a ] ]
    when is_value 1 o && equal a b -> Leaf (Int (-1))
  | Node [ Leaf (Str "*"); a; Node [ Leaf (Str "neg"); Node [ Leaf (Str "/"); o; b ] ] ]
    when is_value 1 o && equal a b -> Leaf (Int (-1))
  | Node [ Leaf (Str "*"); Node [ Leaf (Str "neg"); Node [ Leaf (Str "/"); o; b ] ]; a ]
    when is_value 1 o && equal a b -> Leaf (Int (-1))

  (* Power Derivative Reduction Rules *)
  | Node [ Leaf (Str "*"); Node [ Leaf (Str "*"); a; Node [ Leaf (Str "/"); o; f ] ];
           Node [ Leaf (Str "**"); f2; Leaf ((Int _ | Float _) as b) ] ]
    when is_value 1 o && equal f f2 ->
      if num_eq b (Int 2) then op2 "*" a f else op2 "*" a (op2 "**" f (Leaf (num_sub b (Int 1))))
  | Node [ Leaf (Str "*"); Node [ Leaf (Str "**"); f2; Leaf ((Int _ | Float _) as b) ];
           Node [ Leaf (Str "*"); a; Node [ Leaf (Str "/"); o; f ] ] ]
    when is_value 1 o && equal f f2 ->
      if num_eq b (Int 2) then op2 "*" a f else op2 "*" a (op2 "**" f (Leaf (num_sub b (Int 1))))
  | Node [ Leaf (Str "*"); Node [ Leaf (Str "/"); o; f ]; Node [ Leaf (Str "**"); f2; Leaf ((Int _ | Float _) as b) ] ]
    when is_value 1 o && equal f f2 ->
      if num_eq b (Int 2) then f else op2 "**" f (Leaf (num_sub b (Int 1)))
  | Node [ Leaf (Str "*"); Node [ Leaf (Str "**"); f2; Leaf ((Int _ | Float _) as b) ]; Node [ Leaf (Str "/"); o; f ] ]
    when is_value 1 o && equal f f2 ->
      if num_eq b (Int 2) then f else op2 "**" f (Leaf (num_sub b (Int 1)))

  (* Division *)
  | Node [ Leaf (Str "/"); z; _ ] when is_value 0 z -> zero
  | Node [ Leaf (Str "/"); a; o ] when is_value 1 o -> a
  | Node [ Leaf (Str "/"); a; b ] when equal a b -> one

  (* Exponentiation *)
  | Node [ Leaf (Str "**"); _; z ] when is_value 0 z -> one
  | Node [ Leaf (Str "**"); a; o ] when is_value 1 o -> a
  | Node [ Leaf (Str "**"); z; _ ] when is_value 0 z -> zero
  | Node [ Leaf (Str "**"); o; _ ] when is_value 1 o -> one

  (* Logarithmic & Exponential Identities *)
  | Node [ Leaf (Str "ln"); o ] when is_value 1 o -> zero
  | Node [ Leaf (Str "ln"); Node [ Leaf (Str "exp"); a ] ] -> a
  | Node [ Leaf (Str "exp"); z ] when is_value 0 z -> one
  | Node [ Leaf (Str "exp"); Node [ Leaf (Str "ln"); a ] ] -> a
  | Node [ Leaf (Str "exp"); Node [ Leaf (Str "*"); a; Node [ Leaf (Str "ln"); f ] ] ] -> op2 "**" f a
  | Node [ Leaf (Str "exp"); Node [ Leaf (Str "*"); Node [ Leaf (Str "ln"); f ]; a ] ] -> op2 "**" f a

  (* Fallback *)
  | _ -> e_sim

## String Formatting
The function `precedence` ranks the operators, including the unary operator `neg`.  The function `precedenceOp`
returns the precedence of the operator of an expression; numbers and variables get the highest precedence $5$.

In [ ]:
let precedence (op : string) : int =
  match op with
  | "+" | "-" -> 1
  | "*" | "/" -> 2
  | "neg" -> 3
  | "**" -> 4
  | _ -> 5

let precedenceOp (e : nested) : int =
  match e with
  | Node (Leaf (Str op) :: _) -> precedence op
  | _ -> 5

let rec toString (e : nested) : string =
  let paren cond s = if cond then "(" ^ s ^ ")" else s in
  match e with
  | Leaf (Str s) -> s
  | Leaf v -> Nested.value_to_string v
  | Node [ Leaf (Str "neg"); a ] ->
      (* Only wrap in parens if inner expr is lower precedence than mult/div (+ or -) *)
      if precedenceOp a <= 1 then "-(" ^ toString a ^ ")" else "-" ^ toString a
  | Node [ Leaf (Str f); a ] -> f ^ "(" ^ toString a ^ ")"
  | Node [ Leaf (Str "+"); l; r ] -> toString l ^ " + " ^ toString r
  | Node [ Leaf (Str "-"); l; r ] -> toString l ^ " - " ^ paren (precedenceOp r = 1) (toString r)
  | Node [ Leaf (Str "*"); l; r ] -> paren (precedenceOp l = 1) (toString l) ^ "*" ^ paren (precedenceOp r = 1) (toString r)
  | Node [ Leaf (Str "/"); l; r ] -> paren (precedenceOp l = 1) (toString l) ^ "/" ^ paren (precedenceOp r <= 2) (toString r)
  | Node [ Leaf (Str "**"); l; r ] -> paren (precedenceOp l <= 4) (toString l) ^ "**" ^ paren (precedenceOp r <= 3) (toString r)
  | _ -> invalid_arg ("toString: unexpected tuple " ^ Nested.to_string e)

## Testing

In [ ]:
let test (s : string) : unit =
  let t = parse s in
  let d = simplify (diff t) in
  Printf.printf "d/dx %s = %s\n%!" s (toString d)

In [ ]:
test "x ** x"

In [ ]:
test "x ** -x"

In [ ]:
test "-x ** 2"

In [ ]:
test "1/(1+exp(-x))"

In [ ]:
test "x*x"

In [ ]:
test "x/x"